# ChestAI · Slice-selection

**Goal:** determine whether a small selection of CT slices can preserve informative, non-redundant study evidence, and whether selection can *actually* reduce encoder inference cost.

This notebook is a **development-stage experiment**, not a validated selector. It does not modify preprocessing or production code. Run from the repository's `notebooks/` directory or open it in VS Code. **Read the safeguards before interpreting results.**

### Research questions
1. Do the representations provide a meaningful notion of anatomical difference (and which encoder/pooling is useful)?
2. How do uniform, random, importance, diversity and combined selection behave as K changes?
3. Do methods preserve a full-feature reference representation and abnormality predictions?
4. Can an inexpensive shallow-pass predictor produce useful rankings, and is its measured cost justified?
5. Does a trained classifier on *selected* slices retain held-out predictive performance? (follow-up training required)

**Terminology:** CT **slice**, not slide. **N** = all slices, **K** = selected slices.

## The two experimental tracks and their limits

**Track A — fast retrospective screening:** read existing full-depth `(N,D)` embeddings. Cheap, permits many methods/volumes and paired visual diagnostics. **Not real compute savings:** all N slices have already been fully encoded; metrics such as full-feature coverage are *proxies* and may miss small lesions.

**Track B — live encoder pilot (optional):** process all slices with an early-block, reduced-resolution pass, then full-encode only K selected slices. Measure total cost including preprocessing, selection, the shallow forward pass and final encoding. Compare matched full-encoder settings. This track requires actual HU cache, weights and GPU. Do not call it a win unless end-to-end cost falls *and* downstream task utility holds.

**Avoid leakage:** develop algorithms/hyperparameters using train/validation only. Test set must stay sealed until configuration frozen. No label information is used to choose slices in the implemented selectors. Study labels are used only for stratification/analysis, not for scoring. Patient-level deduplication matters.

**A known limitation:** average representation similarity and feature-space diversity are **not** clinical lesion sensitivity. A rare small lesion can be missed by a mathematically attractive subset. Use held-out per-class AUROC/AUPRC and small-lesion stress tests before drawing claims.

In [ ]:
from pathlib import Path
import os, json, time, warnings, hashlib, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / 'configs' / 'preprocessing.yaml').exists():
    ROOT = ROOT.parent
assert (ROOT / 'src' / 'ct_model').is_dir(), f"Open notebook within the ChestAI repository. Tried {ROOT}"
sys.path.insert(0, str(ROOT / 'src'))

# EDIT ME: paths can be absolute. The ZIP does not contain the large data/ and outputs/ folders.
EMBEDDING_DIR = ROOT / 'data' / 'embeddings'
RUN_MANIFEST = None  # Path('data/runs/ctrate/<run>/manifest.csv') to join split/labels; optional for Track A
STORE_FOLDER = None  # Path to one existing embedding store; None = auto-discover (fails if ambiguous)
OUTPUT = ROOT / 'outputs' / 'slice_selection_lab'
OUTPUT.mkdir(parents=True, exist_ok=True)
SEED = 42
MAX_VOLUMES = 24          # safe exploratory subset, not a final performance evaluation
K_VALUES = [4, 8, 16, 32, 64]
RANDOM_REPEATS = 5
LAMBDA_VALUES = [0.25, 0.50, 0.75]
# methods are intentionally independent of study labels
print('Repository:', ROOT)
print('Output directory:', OUTPUT)
print('Embedding base exists:', EMBEDDING_DIR.is_dir())

## 1 — Safe data discovery and provenance

Load only real, already-computed embeddings; **no synthetic or randomly initialised features** are substituted if the data are missing. Do not mix encoders or HU preprocessing revisions. Auto-discovery refuses an ambiguous store.

In [ ]:
from ct_model.embeddings.store import read_store_record

folders = sorted(d for d in EMBEDDING_DIR.glob('*') if d.is_dir() and (d / '.embedding_record.json').exists()) if EMBEDDING_DIR.exists() else []
display(pd.DataFrame([{'folder': str(d), 'fingerprint': read_store_record(d).get('encoder_fingerprint'), 'dimension': read_store_record(d).get('embed_dim'), 'dtype': read_store_record(d).get('store_dtype')} for d in folders]))
if STORE_FOLDER is None:
    if len(folders) != 1:
        raise FileNotFoundError(f'Expected exactly one embedding store, found {len(folders)}. Set STORE_FOLDER and ensure embeddings exist on disk.')
    STORE_FOLDER = folders[0]
STORE_FOLDER = Path(STORE_FOLDER).expanduser()
record = read_store_record(STORE_FOLDER)
assert record and record.get('embed_dim'), 'Store missing provenance record or dimension'
files = sorted(STORE_FOLDER.glob('*.npy'))
assert files, f'No cached embeddings at {STORE_FOLDER}. Run scripts/model/encode_volumes.py first.'
print(json.dumps(record, indent=2, default=str)[:2500])
print('Available volume embedding arrays:', len(files))

In [ ]:
manifest = pd.read_csv(RUN_MANIFEST, low_memory=False) if RUN_MANIFEST else None
if manifest is not None:
    assert 'volume_id' in manifest and 'split' in manifest
    # Prefer validation for method development, never the held-out test set.
    manifest = manifest[manifest['split'].isin(['train','val'])].drop_duplicates('volume_id')
    if 'qc_passed' in manifest:
        manifest = manifest[manifest['qc_passed'].astype(str).str.lower().isin(['true','1','yes'])]
    eligible = set(manifest.volume_id.astype(str))
    files = [f for f in files if f.stem in eligible]
else:
    warnings.warn('Manifest not set: exploratory features only; no split/patient guarantees. Do not publish generalization scores.')
assert files, 'No eligible cached volumes. Check manifest and embedding IDs.'
rng = np.random.default_rng(SEED)
chosen = [files[i] for i in sorted(rng.choice(len(files), min(MAX_VOLUMES,len(files)), replace=False))]
print('Selected exploratory volumes:', len(chosen))
print('First volumes:', [f.stem for f in chosen[:5]])

## 2 — Slice-selection algorithms

**Methods:** uniform (anatomical coverage), random (repeated control), norm/variation proxy (naïve importance), farthest-first cosine diversity (coverage), importance + greedy diversity (λ sweep), and spatially stratified importance (guards against selecting only one region).

Importance here is **a hypothesis**, NOT a proven pathology score: norm/feature variation may highlight noise, acquisition effects, bones or unusual anatomy. In full-depth Track A it is merely a *retrospective oracle-space proxy*. In Track B we apply a different cheap early-block proxy. Selecting top-K highest raw feature norm is not assumed to diagnose a lesion.

In [ ]:
def normalize_rows(x):
    x = np.asarray(x, dtype=np.float32)
    return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-8)

def minmax(x):
    x = np.asarray(x, dtype=np.float32)
    return (x-x.min()) / max(float(np.ptp(x)), 1e-8)

def importance_score(x, mode='deviation'):
    x = np.asarray(x, dtype=np.float32)
    if mode == 'norm':
        return minmax(np.linalg.norm(x,axis=1))
    # departure from the study's median representation (hypothesis, not pathology score)
    ref = np.median(x,axis=0)
    return minmax(np.linalg.norm(x-ref,axis=1))

def uniform_indices(n,k):
    return np.unique(np.rint(np.linspace(0,n-1,min(k,n))).astype(int))

def choose_slices(x, k, method, *, seed=SEED, lam=0.5, importance=None):
    x = np.asarray(x, dtype=np.float32)
    if x.ndim != 2 or len(x) == 0 or not np.isfinite(x).all():
        raise ValueError('Expected finite (N,D) embeddings')
    n = len(x); k = min(max(1,int(k)),n)
    if k == n: return np.arange(n)
    if method == 'uniform': return uniform_indices(n,k)
    if method == 'random': return np.sort(np.random.default_rng(seed).choice(n,k,replace=False))
    imp = importance_score(x) if importance is None else minmax(importance)
    if len(imp) != n: raise ValueError('Importance length mismatch')
    if method == 'importance': return np.sort(np.argsort(-imp,kind='stable')[:k])
    if method == 'stratified_importance':
        # one highest-score slice from each of K contiguous partitions
        bins = np.array_split(np.arange(n),k)
        return np.array(sorted(int(b[np.argmax(imp[b])]) for b in bins))
    if method not in ('diversity','combined'):
        raise ValueError(method)
    z = normalize_rows(x)
    # cosine distance, can be negative similarity; clamp for floating-point stability
    chosen = [int(np.argmax(imp))] if method == 'combined' else [int(n//2)]
    closest = 1.0 - z @ z[chosen[0]]
    closest[chosen[0]] = -np.inf
    for _ in range(1,k):
        # max-min: candidate's min distance to its nearest already chosen neighbor
        div = minmax(np.where(np.isfinite(closest),closest,0.0))
        score = div if method == 'diversity' else lam*imp + (1-lam)*div
        score[chosen] = -np.inf
        j = int(np.argmax(score))
        chosen.append(j)
        closest = np.minimum(closest,1.0-z@z[j]); closest[chosen] = -np.inf
    return np.array(sorted(chosen))

METHODS = ['uniform','random','importance','stratified_importance','diversity'] + [f'combined_{v:.2f}' for v in LAMBDA_VALUES]
def get_method(name):
    if name.startswith('combined_'): return 'combined',float(name.split('_')[-1])
    return name,0.5
# Invariants: bounded, unique, correct budget, repeatable, no leakage from label information
for method in METHODS:
    fake = np.random.default_rng(0).normal(size=(60,12)).astype('float32')
    name,lam = get_method(method)
    for k in (1,8,60,70):
        ids = choose_slices(fake,k,name,seed=17,lam=lam)
        assert len(ids)==min(k,60) and len(set(ids))==len(ids)
        assert min(ids)>=0 and max(ids)<60
        assert np.array_equal(ids,choose_slices(fake,k,name,seed=17,lam=lam))
print('Selector invariants passed.')

## 3 — Quantitative proxy diagnostics

**Coverage:** cosine distance from each *full-depth embedding* to its nearest selected embedding (lower better). **Representational drift:** cosine distance between full-volume and selected-volume mean embeddings (lower better). **Redundancy:** mean selected pairwise similarity (lower generally indicates diversity). **Axial gap:** largest unselected gap in slice indices (lower indicates anatomical coverage). These can disagree; none alone proves preservation of diagnostic abnormalities.

Reference is *full-volume representation*, not ground-truth lesion location. We also check a **small focal signal counterexample** below.

In [ ]:
def proxy_metrics(full, idx):
    f = normalize_rows(full); s=f[idx]
    nearest = np.max(f@s.T,axis=1)
    dense = np.asarray(full,dtype=np.float32)
    a = dense.mean(axis=0); b=dense[idx].mean(axis=0)
    drift = 1.0 - float(a@b / max(np.linalg.norm(a)*np.linalg.norm(b),1e-8))
    triu = (s@s.T)[np.triu_indices(len(s),1)]
    gap = np.diff(np.r_[-1,idx,len(full)])-1
    return {'nearest_cosine_distance':float(np.mean(1-nearest)),
            'p95_cosine_distance':float(np.percentile(1-nearest,95)),
            'mean_embedding_drift':drift,
            'selected_redundancy':float(triu.mean()) if len(triu) else np.nan,
            'largest_axial_gap':int(max(gap)),
            'fraction_encoded':len(idx)/len(full)}

rows=[]
for f in chosen:
    emb=np.load(f,mmap_mode='r')
    if emb.ndim != 2 or emb.shape[1] != record['embed_dim'] or not np.isfinite(emb).all():
        warnings.warn(f'Skipping bad embeddings {f.name}'); continue
    for k in K_VALUES:
        for method in METHODS:
            m,lam=get_method(method)
            for rep in range(RANDOM_REPEATS if m=='random' else 1):
                idx=choose_slices(emb,k,m,seed=SEED+rep,lam=lam)
                rows.append(dict(volume_id=f.stem,n_slices=len(emb),k=min(k,len(emb)),method=method,repeat=rep,
                                 **proxy_metrics(emb,idx)))
results=pd.DataFrame(rows)
assert not results.empty, 'No usable embedding arrays'
summary=results.groupby(['method','k'],as_index=False)[['nearest_cosine_distance','p95_cosine_distance','mean_embedding_drift','selected_redundancy','largest_axial_gap','fraction_encoded']].mean()
display(summary.sort_values(['k','nearest_cosine_distance']).head(30))
summary.to_csv(OUTPUT/'proxy_summary.csv',index=False)
results.to_csv(OUTPUT/'proxy_per_volume.csv',index=False)
print('Proxy results saved; these are NOT predictive metrics.')

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(13,4.5))
for method,df in summary.groupby('method'):
    axs[0].plot(df.k,df.nearest_cosine_distance,marker='o',label=method)
    axs[1].plot(df.k,df.mean_embedding_drift,marker='o',label=method)
axs[0].set(title='Full-feature coverage proxy (lower is better)',xlabel='K slices',ylabel='Mean nearest cosine distance')
axs[1].set(title='Mean-representation drift proxy (lower is better)',xlabel='K slices',ylabel='Cosine distance')
axs[1].legend(bbox_to_anchor=(1.02,1),loc='upper left'); plt.tight_layout(); plt.show()

## 4 — Inspect selected axial positions and actual CT images

Select a scan and K, compare different selection patterns. If the original `data/cache/{volume_id}.npy` exists, display selected CT slices using lung window. These pictures are **for human inspection**, not per-slice ground truth or proof of detecting pathology.

In [ ]:
VOLUME_ID=chosen[0].stem
VIEW_K=16
E=np.load(STORE_FOLDER/f'{VOLUME_ID}.npy',mmap_mode='r')
fig,ax=plt.subplots(figsize=(14,4))
for i,name in enumerate(METHODS):
    method,lam=get_method(name)
    idx=choose_slices(E,VIEW_K,method,lam=lam)
    ax.scatter(idx,np.full(len(idx),i),marker='|',s=160,label=name)
ax.set(yticks=range(len(METHODS)),yticklabels=METHODS,xlabel='Axial slice index',title=f'Selected positions · {VOLUME_ID} · N={len(E)}')
ax.grid(axis='x',alpha=.2); plt.tight_layout(); plt.show()
CACHE_DIR = ROOT/'data'/'cache'
HU_PATH=CACHE_DIR/f'{VOLUME_ID}.npy'
if HU_PATH.exists():
    HU=np.load(HU_PATH,mmap_mode='r')
    method,lam=get_method('combined_0.50')
    ids=choose_slices(E,min(8,len(E)),method,lam=lam)
    fig,axs=plt.subplots(1,len(ids),figsize=(2.1*len(ids),2.6))
    for ax,j in zip(np.atleast_1d(axs),ids):
        ax.imshow(np.clip((HU[j]+1000)/1400,0,1),cmap='gray',vmin=0,vmax=1)
        ax.set_title(f'z={j}');ax.axis('off')
    plt.tight_layout();plt.show()
else: print('HU cache unavailable; position plot still valid:',HU_PATH)

## 5 — Failure-mode stress test: a tiny focal abnormality

This deliberately simple synthetic feature-space test **does not simulate real CT images**. It asks whether an otherwise almost identical sequence with a **single distinctive feature vector** gets selected. Contrast different methods. Repeat with smaller signal amplitude, noise, and different axial positions. A method can have good average coverage and still fail on the clinically important one-slice case.

In [ ]:
def focal_stress(trials=60,n=180,d=48,k=16,signal=0.8,noise=0.04):
    out=[]; rg=np.random.default_rng(2026)
    for t in range(trials):
        base=rg.normal(size=d).astype(np.float32)
        X=base+rg.normal(scale=noise,size=(n,d)).astype(np.float32)
        focal=int(rg.integers(5,n-5)); direction=rg.normal(size=d);direction/=np.linalg.norm(direction)
        X[focal]+=signal*np.linalg.norm(base)*direction
        for name in METHODS:
            method,lam=get_method(name)
            idx=choose_slices(X,k,method,seed=SEED+t,lam=lam)
            out.append({'method':name,'hit':int(focal in idx),'nearest_gap':int(np.min(abs(idx-focal)))})
    return pd.DataFrame(out).groupby('method',as_index=False).agg(hit_rate=('hit','mean'),mean_axial_miss=('nearest_gap','mean'))
display(focal_stress().sort_values('hit_rate',ascending=False))
print('WARNING: artificial embeddings only; never report this as clinical sensitivity.')

## 6 — Are two encoders useful in the same sense?

**Encoder selection cannot be decided by self-similarity alone.** DALE-CT-2S is CT-pretrained; the alternative DINOv2 config uses three HU windows and natural-image pretraining. Compare representation coverage, compute cost, slice ranks, and **held-out downstream multi-label metrics with a matched classifier protocol**. The current ZIP contains configs but no DINOv2 embedding arrays.

Below is a reproducible *paired-store* proxy comparison. Set `SECOND_STORE` only when it exists. The same volume IDs, splits and K must be used. Raw cosine distance values between architectures need careful interpretation: differences in embedding geometry can make raw numeric coverage scores incomparable. Within-encoder method rankings are more informative.

In [ ]:
SECOND_STORE = None  # e.g. ROOT/'data/embeddings/dinov2_vitl14-xxxxxxxx'
if SECOND_STORE:
    secondary=Path(SECOND_STORE)
    assert read_store_record(secondary), 'Missing second store provenance'
    matches=[f for f in chosen if (secondary/f.name).exists()]
    compare=[]
    for f in matches:
        for tag,folder in [('primary',STORE_FOLDER),('secondary',secondary)]:
            x=np.load(folder/f.name,mmap_mode='r')
            if len(x)!=len(np.load(STORE_FOLDER/f.name,mmap_mode='r')): raise ValueError('Slice count mismatch')
            for name in METHODS:
                method,lam=get_method(name); ids=choose_slices(x,min(16,len(x)),method,lam=lam)
                compare.append({'encoder':tag,'method':name,'volume_id':f.stem,**proxy_metrics(x,ids)})
    display(pd.DataFrame(compare).groupby(['encoder','method'])[['nearest_cosine_distance','mean_embedding_drift']].mean())
else: print('Skipped second-store comparison. Create second encoder embeddings through existing encode_volumes.py.')

## 7 — Optional live shallow-encoder feasibility pilot (GPU recommended)

Run only after Track A. This section calls the existing **frozen** `TimmViTEncoder`; it does not rewrite preprocessing or modify the model. We patch only the exploratory forward path by explicitly using the model's patch-embedding, positional embedding and first transformer blocks, then extracting a cheap per-slice proxy. The `timm` internal API may differ by version; assertions fail loudly instead of silently returning fabricated features.

**Fair cost comparison:** use identical slice count, full-encoder resolution, warmup and device; count the shallow pass on **all N slices**, selection CPU overhead and full encoding of K. Synchronize CUDA when timing. As input preprocessing is already cached, report the measurement as **from HU-cache to embeddings**, not raw DICOM-to-result. Original DALE was trained at 512 but repository uses 224; use the configured transform and do not mistake reduced scoring resolution for validated full-resolution input.

**Warning:** raw shallow tokens have no guaranteed correlation with pathology. This is a feasibility pilot, not the final selector.

In [ ]:
RUN_LIVE_PILOT=False   # True only with actual HU cache, working weights, memory and GPU
ENCODER_NAME='dale_ct_2s'
EARLY_BLOCKS=2
SHALLOW_SIZE=112            # DALE 16-pixel patches; 112 is divisible by 16
LIVE_K=16
LIVE_VOLUME_ID=VOLUME_ID
LIVE_BATCH=8

def sync_cuda(device):
    if device.type=='cuda':
        import torch
        torch.cuda.synchronize(device)

if RUN_LIVE_PILOT:
    import torch
    import torch.nn.functional as F
    from ct_model.config import load_encoder_config
    from ct_model.encoders.registry import build_encoder
    cfg=load_encoder_config(ENCODER_NAME)
    assert SHALLOW_SIZE%cfg.model_args.get('patch_size',16)==0
    encoder=build_encoder(cfg,frozen=True)
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    encoder.to(device).eval()
    hu_file=CACHE_DIR/f'{LIVE_VOLUME_ID}.npy'
    assert hu_file.exists(),f'Missing HU cache: {hu_file}'
    volume=np.load(hu_file,mmap_mode='r')
    assert volume.ndim==3 and len(volume)>LIVE_K
    def early_features(batch_hu):
        x=encoder.input_transform(batch_hu)
        x=F.interpolate(x,size=(SHALLOW_SIZE,SHALLOW_SIZE),mode='bilinear',align_corners=False)
        model=encoder.model
        x=model.patch_embed(x)
        x=model._pos_embed(x)  # timm handles dynamic grid, CLS/prefix tokens and interpolation
        x=model.patch_drop(x)
        x=model.norm_pre(x)
        for block in model.blocks[:EARLY_BLOCKS]:
            x=block(x)
        x=x[:,model.num_prefix_tokens:]
        # mean shallow patch token + patchwise standard-deviation scalar as importance signal
        return x.mean(dim=1), x.std(dim=1).mean(dim=1)
    def time_call(fn):
        sync_cuda(device);t=time.perf_counter();out=fn();sync_cuda(device)
        return out,time.perf_counter()-t
    def all_batches(fn,indices):
        out=[];scores=[]
        with torch.inference_mode():
            for chunk in np.array_split(np.asarray(indices),int(np.ceil(len(indices)/LIVE_BATCH))):
                tensor=torch.from_numpy(np.asarray(volume[chunk],dtype=np.float32)).to(device)
                feat=fn(tensor)
                if isinstance(feat,tuple):
                    out.append(feat[0].float().cpu().numpy());scores.append(feat[1].float().cpu().numpy())
                else: out.append(feat.float().cpu().numpy())
        return np.concatenate(out),np.concatenate(scores) if scores else None
    # Warmup kernels and model weights before timing.
    _=all_batches(lambda v:encoder(v),np.arange(min(LIVE_BATCH,len(volume))))
    _=all_batches(early_features,np.arange(min(LIVE_BATCH,len(volume))))
    shallow,t_shallow=time_call(lambda:all_batches(early_features,np.arange(len(volume))))
    shallow_vectors,shallow_importance=shallow
    def selected_ids():
        return choose_slices(shallow_vectors,LIVE_K,'combined',lam=.5,importance=shallow_importance)
    idx,t_selection=time_call(selected_ids)
    (picked,_),t_full_k=time_call(lambda:all_batches(lambda v:encoder(v),idx))
    (baseline,_),t_full_n=time_call(lambda:all_batches(lambda v:encoder(v),np.arange(len(volume))))
    report_live=dict(encoder=ENCODER_NAME,volume_id=LIVE_VOLUME_ID,n=len(volume),k=len(idx),early_blocks=EARLY_BLOCKS,
      shallow_size=SHALLOW_SIZE,seconds_shallow=t_shallow,seconds_selection=t_selection,
      seconds_full_k=t_full_k,seconds_full_n=t_full_n,
      total_selected_seconds=t_shallow+t_selection+t_full_k,
      speedup=(t_full_n/(t_shallow+t_selection+t_full_k)),device=str(device))
    display(pd.DataFrame([report_live]));(OUTPUT/'live_pilot.json').write_text(json.dumps(report_live,indent=2))
    print('Selected slices:',idx)
else:
    print('Live shallow pilot disabled. Set RUN_LIVE_PILOT=True after confirming GPU, weights and HU cache.')

## 8 — From proxy research to actual predictive validation

**What this notebook establishes:** computationally testable candidate selectors, baselines, selection positions, full-feature proxy retention, focal-stress failure modes, alternative encoder *comparison hooks* and optional measured shallow-encoder feasibility.

**What it cannot establish without additional runs:** multi-label predictive preservation and fairness under actual reduced-depth encoding. Those require:

1. Freeze one candidate selector, K and λ using **development training/validation**, not test.
2. On a **matched patient-level split**, train/retrain the *same aggregation/classification architecture* separately under each selector (or do a separately-labelled frozen-classifier diagnostic). Keep random seeds, optimiser, thresholds, preprocessing and encoder consistent.
3. Evaluate **per-label AUROC, AUPRC**, macro/micro scores, small-lesion categories, scanner/kernel strata and patient-bootstrap confidence intervals. Include *uniform*, *random repeated*, *importance-only*, *diversity-only*, *combined* and *full N*.
4. Include selection overhead and repeat warm timings; measure VRAM, batches, wall time and, if possible, power/energy. For cached full embeddings, do **not** count downstream indexing as inference savings.
5. Inspect actual missed findings with clinician review where available. Specifically assess tiny nodules and peripherally localised abnormalities.
6. Only after choosing methods/thresholds run once on the **sealed test** cohort. Save exact model/config fingerprints, manifest identifiers, environment and measurements.

**Causality:** comparisons that use the same downstream model frozen after full-volume training tell you robustness to dropped evidence, not how well a new reduced-budget model can be trained. Report those as different experiments. If only the full-depth cache is available, the result is hypothesis screening—not a demonstration of an efficient selector.

**Integration target after evidence supports a winner:** create an independent `SliceSelector` interface consuming HU volume / cheap features and yielding sorted, unique slice indices; preserve existing `ct_preprocessing` unmodified. Wire selector **before** expensive `SliceEncoder` within a new embedding extraction pathway, carrying selected index metadata into the embedding store and evidence pipeline. Never quietly overwrite the current full-volume embedding cache: the store identity and slice-index mapping must change.

In [ ]:
metadata={'purpose':'development-stage selection research', 'seed':SEED,
          'store':str(STORE_FOLDER),'store_record':record,'k_values':K_VALUES,
          'lambdas':LAMBDA_VALUES,'methods':METHODS,'n_volumes':len(chosen),
          'manifest':str(RUN_MANIFEST) if RUN_MANIFEST else None,
          'warning':'Full-depth cached embeddings are proxy diagnostics; no inference-time savings established.'}
(OUTPUT/'run_metadata.json').write_text(json.dumps(metadata,indent=2,default=str))
print('Saved artifacts:', sorted(x.name for x in OUTPUT.iterdir()))
print('Next: inspect failures, then validate classifier task metrics and live selector timings.')